# A. 고객 세분화 & 페르소나

고객별 구매 횟수·결제액·선호 카테고리·결제 수단·지역을 집계한 뒤 **K-Means 군집**으로 세그먼트를 나누고, 세그먼트별 페르소나를 요약·시각화합니다.

**사용 데이터**: `00-Olist_데이터마트_전처리.ipynb` 실행 후 `data/` 폴더.

---

**전체 vs 시기별**
- **현재 군집**: **전체 기간(2016~2018) 데이터**를 한 번에 모아서 고객당 하나의 세그먼트를 부여합니다. 즉 "전체 데이터 기준" 한 번의 세분화입니다.
- **시기별로 확인**하려면: (1) 같은 세그먼트를 유지한 채 **첫 구매 연도/분기**로 잘라서 "연도별 세그먼트 구성"을 보거나, (2) **연도·분기별로 따로 군집**을 돌려 시기마다 다른 세그먼트 정의를 쓸 수 있습니다. 아래 "5.5 시기별 확인"에서 (1)을, 필요 시 (2)는 연도별로 동일 코드를 반복해 적용하면 됩니다.

## 1. 데이터 로드

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# data/ 폴더 경로 (여러 위치 시도)
DATA_DIR = Path.cwd() / "data"
for candidate in [Path.cwd() / "data", Path.cwd() / "예측 대시보드 용 프로젝트" / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (candidate / "customers.csv").exists():
        DATA_DIR = candidate
        break
if not (DATA_DIR / "customers.csv").exists():
    # data/ 없으면 kagglehub에서 직접 로드 + 최소 전처리
    import kagglehub
    _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    customers = pd.read_csv(_path / "olist_customers_dataset.csv")
    orders = pd.read_csv(_path / "olist_orders_dataset.csv")
    order_items = pd.read_csv(_path / "olist_order_items_dataset.csv")
    products = pd.read_csv(_path / "olist_products_dataset.csv")
    order_payments = pd.read_csv(_path / "olist_order_payments_dataset.csv")
    products = products.rename(columns={"product_name_lenght": "product_name_length", "product_description_lenght": "product_description_length"})
    products["product_category_name"] = products["product_category_name"].fillna("unknown")
    order_payments["payment_type"] = order_payments["payment_type"].replace("not_defined", "unknown")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    orders = orders[orders["order_status"] == "delivered"].copy()
    print("(data/ 없음 → kagglehub에서 로드)")
else:
    customers = pd.read_csv(DATA_DIR / "customers.csv")
    orders = pd.read_csv(DATA_DIR / "orders_delivered.csv")
    order_items = pd.read_csv(DATA_DIR / "order_items.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    order_payments = pd.read_csv(DATA_DIR / "order_payments.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

print("customers:", customers.shape, "| orders:", orders.shape)

customers: (99441, 5) | orders: (96478, 8)


## 2. 고객별 집계 (customer_unique_id 기준)

In [2]:
# 주문-상품-카테고리 결합
ord_items = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
ord_items["product_category_name"] = ord_items["product_category_name"].fillna("unknown")
ord = orders[["order_id", "customer_id", "order_purchase_timestamp"]].merge(ord_items, on="order_id").merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")

# 주문 금액 (order_items 기준 합계)
order_total = ord.groupby("order_id").agg(price=("price", "sum"), freight_value=("freight_value", "sum")).reset_index()
order_total["order_value"] = order_total["price"] + order_total["freight_value"]
ord = ord.merge(order_total[["order_id", "order_value"]], on="order_id")

# 고객별 1주문당 1행이 되도록 order 기준
ord_cust = ord.drop_duplicates(subset=["order_id", "customer_unique_id"])

# 고객별 지표 (첫 주문 시점 포함 → 시기별 분석용)
cust_orders = ord_cust.groupby("customer_unique_id").agg(
    order_count=("order_id", "nunique"),
    total_value=("order_value", "sum"),
    customer_state=("customer_state", "first"),
    first_order_date=("order_purchase_timestamp", "min"),
).reset_index()
cust_orders["avg_order_value"] = cust_orders["total_value"] / cust_orders["order_count"]

# 카테고리별 결제액 (고객별) — apply 제거, list comprehension + merge로 속도 개선
cat_sum = ord.groupby(["customer_unique_id", "product_category_name"])["price"].sum()
top_cat_series = cat_sum.groupby(level=0).idxmax()
top_cat_df = pd.DataFrame({"customer_unique_id": top_cat_series.index, "top_category": [x[1] for x in top_cat_series.values]})
cust_orders = cust_orders.merge(top_cat_df, on="customer_unique_id", how="left")

# 결제 수단 (고객별 최빈값) — mode() 대신 value_counts idxmax로 속도 개선
pay_first = order_payments.groupby("order_id")["payment_type"].first().reset_index()
ord_pay = ord_cust[["order_id", "customer_unique_id"]].merge(pay_first, on="order_id")
pay_counts = ord_pay.groupby(["customer_unique_id", "payment_type"]).size().reset_index(name="n")
pay_mode = pay_counts.loc[pay_counts.groupby("customer_unique_id")["n"].idxmax()][["customer_unique_id", "payment_type"]].rename(columns={"payment_type": "main_payment_type"})
cust_orders = cust_orders.merge(pay_mode, on="customer_unique_id", how="left")
cust_orders["first_order_year"] = pd.to_datetime(cust_orders["first_order_date"]).dt.year
cust_orders["first_order_week_start"] = pd.to_datetime(cust_orders["first_order_date"]).dt.to_period("W-MON").dt.start_time

print(cust_orders.head())

                 customer_unique_id  order_count  total_value customer_state  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1      141.900             SP   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1       27.190             SP   
2  0000f46a3911fa3c0805444483337064            1       86.220             SC   
3  0000f6ccb0745a6a4b88665a16c9f078            1       43.620             PA   
4  0004aac84e0df4da2b147fca70cf8255            1      196.890             SP   

     first_order_date  avg_order_value     top_category main_payment_type  \
0 2018-05-10 10:56:27          141.900  cama_mesa_banho       credit_card   
1 2018-05-07 11:11:27           27.190     beleza_saude       credit_card   
2 2017-03-10 21:05:03           86.220        papelaria       credit_card   
3 2017-10-12 20:29:41           43.620        telefonia       credit_card   
4 2017-11-14 19:45:42          196.890        telefonia       credit_card   

   first_order_year  
0              2018  
1           

## 3. 군집용 피처 구성 및 스케일링

In [3]:
# 수치형만 사용 (state, top_category, main_payment_type은 군집 후 해석용)
X = cust_orders[["order_count", "total_value", "avg_order_value"]].copy()
X["order_count"] = np.log1p(X["order_count"])
X["total_value"] = np.log1p(X["total_value"])
X["avg_order_value"] = np.log1p(X["avg_order_value"])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

## 4. K-Means 군집 (실루엣으로 k 선택)

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 11)
sil_scores = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, lab))

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(ks, sil_scores, "o-")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("K-Means k 선택")
plt.grid(True)
plt.show()

best_k = ks[np.argmax(sil_scores)]
print("선택 k:", best_k)

c:\Users\itwill\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\itwill\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 268, in _count_physical_cores
    cpu_count_physical = sum(map(int, cpu_info))
                         ^^^^^^^^^^^^^^^^^^^^^^^


MemoryError: Unable to allocate 1.00 GiB for an array with shape (1437, 93358) and data type float64

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cust_orders["segment"] = km.fit_predict(X_scaled)
cust_orders["segment"] = "S" + cust_orders["segment"].astype(str)

## 5. 세그먼트별 페르소나 요약

In [ ]:
summary = cust_orders.groupby("segment").agg(
    고객수=("customer_unique_id", "count"),
    평균_주문수=("order_count", "mean"),
    평균_총결제액=("total_value", "mean"),
    평균_주문당결제액=("avg_order_value", "mean"),
    대표_결제수단=("main_payment_type", lambda x: x.mode().iloc[0]),
    대표_카테고리=("top_category", lambda x: x.mode().iloc[0]),
).round(2)
print(summary)

print("\n세그먼트별 state 상위 3:")
for seg in cust_orders["segment"].unique():
    top_states = cust_orders[cust_orders["segment"] == seg]["customer_state"].value_counts().head(3)
    print(seg, top_states.tolist())

## 5.5 시기별로 세그먼트 확인

같은 **전체 기준 세그먼트**를 유지한 채, **첫 구매 연도**로 잘라서 "연도별로 세그먼트 구성이 어떻게 다른지" 볼 수 있습니다. (Looker에서는 `first_order_year` 필터로 동일하게 자를 수 있습니다.)

In [ ]:
# 연도별 세그먼트 구성 (고객 수)
year_segment = pd.crosstab(cust_orders["first_order_year"], cust_orders["segment"])
year_segment_pct = year_segment.div(year_segment.sum(axis=1), axis=0) * 100
print("연도별 세그먼트 고객 수:")
print(year_segment)
print("\n연도별 세그먼트 비율(%):")
print(year_segment_pct.round(1))

# 시각화: 연도별 세그먼트 비율 (스택 바)
year_segment_pct.plot(kind="bar", stacked=True, figsize=(10, 4), colormap="tab10")
plt.title("첫 구매 연도별 세그먼트 구성 (%)")
plt.xlabel("첫 구매 연도")
plt.legend(title="세그먼트", bbox_to_anchor=(1.02, 1))
plt.tick_params(axis="x", rotation=0)
plt.show()

## 5.5b 주별 세그먼트 요약 (대시보드용)

**first_order_week_start** 기준으로 주별·세그먼트별 고객 수·주문 수·매출을 집계합니다. Looker 등에서 주 단위로 슬라이스할 수 있습니다.

In [ ]:
weekly_segment = cust_orders.groupby(["first_order_week_start", "segment"]).agg(
    customer_count=("customer_unique_id", "nunique"),
    order_count=("order_count", "sum"),
    total_value=("total_value", "sum"),
).reset_index()
weekly_segment["year_week"] = pd.to_datetime(weekly_segment["first_order_week_start"]).dt.isocalendar().year.astype(str) + "-W" + pd.to_datetime(weekly_segment["first_order_week_start"]).dt.isocalendar().week.astype(str).str.zfill(2)
print("주별 세그먼트 요약 (대시보드용):")
print(weekly_segment.head(15))
print("\n총 주 수:", weekly_segment["first_order_week_start"].nunique())

## 5.6 페르소나 시각화

세그먼트별 **고객 수**, **주문/결제 특성**, **대표 카테고리·결제수단**을 차트로 그려 페르소나를 한눈에 볼 수 있습니다. (Looker Studio에서도 동일한 축으로 페이지를 구성하면 됩니다.)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (1) 세그먼트별 고객 수
seg_counts = cust_orders["segment"].value_counts().sort_index()
axes[0, 0].bar(seg_counts.index, seg_counts.values, color="steelblue", edgecolor="black")
axes[0, 0].set_title("세그먼트별 고객 수")
axes[0, 0].set_xlabel("세그먼트")

# (2) 세그먼트별 평균 지표 (정규화 바)
seg_metrics = cust_orders.groupby("segment")[["order_count", "total_value", "avg_order_value"]].mean()
seg_metrics_norm = (seg_metrics - seg_metrics.min()) / (seg_metrics.max() - seg_metrics.min() + 1e-8)
seg_metrics_norm.plot(kind="bar", ax=axes[0, 1], width=0.8)
axes[0, 1].set_title("세그먼트별 평균 지표 (0~1 정규화)")
axes[0, 1].set_xlabel("세그먼트")
axes[0, 1].legend(["주문 수", "총 결제액", "주문당 결제액"])
axes[0, 1].tick_params(axis="x", rotation=0)

# (3) 세그먼트별 대표 카테고리 분포 (상위 5개)
top_cats = cust_orders["top_category"].value_counts().head(5).index.tolist()
cat_by_seg = cust_orders[cust_orders["top_category"].isin(top_cats)].groupby(["segment", "top_category"]).size().unstack(fill_value=0)
cat_by_seg.plot(kind="bar", ax=axes[1, 0], width=0.8)
axes[1, 0].set_title("세그먼트별 선호 카테고리 (상위 5개)")
axes[1, 0].set_xlabel("세그먼트")
axes[1, 0].legend(title="카테고리", bbox_to_anchor=(1.02, 1), fontsize=8)
axes[1, 0].tick_params(axis="x", rotation=0)

# (4) 세그먼트별 결제 수단 분포
pay_by_seg = pd.crosstab(cust_orders["segment"], cust_orders["main_payment_type"])
pay_by_seg_pct = pay_by_seg.div(pay_by_seg.sum(axis=1), axis=0) * 100
pay_by_seg_pct.plot(kind="bar", stacked=True, ax=axes[1, 1], colormap="Set3")
axes[1, 1].set_title("세그먼트별 결제 수단 비율 (%)")
axes[1, 1].set_xlabel("세그먼트")
axes[1, 1].legend(title="결제 수단", bbox_to_anchor=(1.02, 1))
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 6. 분기별 군집 — 세그먼트가 어떻게 바뀌어 갔는지

각 **분기(quarter)** 데이터만으로 따로 K-Means를 돌려, 시기별로 세그먼트 **구성 비율·규모**가 어떻게 달라졌는지 봅니다.  
(분기마다 군집이 새로 잡히므로, "S0"가 Q1에서는 저가형, Q2에서는 다른 성격일 수 있습니다. **비율·추이** 비교에 초점을 두면 됩니다.)

In [ ]:
# 분기 컬럼 추가 (위에서 만든 ord 사용)
ord["order_quarter"] = pd.to_datetime(ord["order_purchase_timestamp"]).dt.to_period("Q").dt.to_timestamp()
quarters = sorted(ord["order_quarter"].dropna().unique().tolist())
# 고객 수가 너무 적은 분기 제외 (선택)
min_customers = 2000
from sklearn.preprocessing import StandardScaler

quarter_results = []
for q in quarters:
    ord_q = ord[ord["order_quarter"] == q]
    ord_cust_q = ord_q.drop_duplicates(subset=["order_id", "customer_unique_id"])
    n_cust = ord_cust_q["customer_unique_id"].nunique()
    if n_cust < min_customers:
        continue
    cust_q = ord_cust_q.groupby("customer_unique_id").agg(
        order_count=("order_id", "nunique"),
        total_value=("order_value", "sum"),
    ).reset_index()
    cust_q["avg_order_value"] = cust_q["total_value"] / cust_q["order_count"]
    cat_sum_q = ord_q.groupby(["customer_unique_id", "product_category_name"])["price"].sum()
    top_idx_q = cat_sum_q.groupby(level=0).idxmax()
    top_cat_df_q = pd.DataFrame({"customer_unique_id": top_idx_q.index, "top_category": [x[1] for x in top_idx_q.values]})
    cust_q = cust_q.merge(top_cat_df_q, on="customer_unique_id", how="left")
    pay_first = order_payments.groupby("order_id")["payment_type"].first().reset_index()
    ord_pay_q = ord_cust_q[["order_id", "customer_unique_id"]].merge(pay_first, on="order_id")
    pay_cnt_q = ord_pay_q.groupby(["customer_unique_id", "payment_type"]).size().reset_index(name="n")
    pay_mode_q = pay_cnt_q.loc[pay_cnt_q.groupby("customer_unique_id")["n"].idxmax()][["customer_unique_id", "payment_type"]].rename(columns={"payment_type": "main_payment_type"})
    cust_q = cust_q.merge(pay_mode_q, on="customer_unique_id", how="left")
    X_q = cust_q[["order_count", "total_value", "avg_order_value"]].copy()
    X_q = np.log1p(X_q)
    scaler_q = StandardScaler()
    X_q_scaled = scaler_q.fit_transform(X_q)
    km_q = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    cust_q["segment"] = "S" + km_q.fit_predict(X_q_scaled).astype(str)
    cust_q["quarter"] = q
    quarter_results.append(cust_q[["quarter", "customer_unique_id", "segment", "order_count", "total_value", "avg_order_value"]])

if not quarter_results:
    print("조건을 만족하는 분기가 없습니다. min_customers를 낮춰 보세요.")
else:
    by_quarter = pd.concat(quarter_results, ignore_index=True)
    print("분기별 군집 완료. 분기 수:", by_quarter["quarter"].nunique())

In [ ]:
# 분기 × 세그먼트 구성 (고객 수, 비율)
if quarter_results:
    qseg = pd.crosstab(by_quarter["quarter"], by_quarter["segment"])
    qseg_pct = qseg.div(qseg.sum(axis=1), axis=0) * 100
    print("분기별 세그먼트 고객 수:")
    print(qseg)
    print("\n분기별 세그먼트 비율 (%):")
    print(qseg_pct.round(1))
    qseg_pct.plot(kind="bar", stacked=True, figsize=(12, 4), colormap="tab10")
    plt.title("분기별 세그먼트 구성 (%) — 시기마다 군집이 새로 잡힘")
    plt.xlabel("분기")
    plt.legend(title="세그먼트", bbox_to_anchor=(1.02, 1))
    plt.tick_params(axis="x", rotation=45)
    plt.show()

In [ ]:
# 분기·세그먼트별 평균 지표 (세그먼트 성격이 시기마다 어떻게 달라졌는지)
if quarter_results:
    qseg_profile = by_quarter.groupby(["quarter", "segment"]).agg(
        고객수=("customer_unique_id", "count"),
        평균_주문수=("order_count", "mean"),
        평균_총결제액=("total_value", "mean"),
        평균_주문당결제액=("avg_order_value", "mean"),
    ).round(2)
    print(qseg_profile)